# Python OOP

Four pillars: **Encapsulation**, **Inheritance**, **Polymorphism**, **Abstraction**.

| Concept       | What it means                                            | Python tool                          |
|---------------|----------------------------------------------------------|--------------------------------------|
| Encapsulation | Bundle data + behavior; hide internals                   | `class`, `_protected`, `__private`   |
| Inheritance   | Child class reuses/extends parent                        | `class Child(Parent):`, `super()`    |
| Polymorphism  | Same interface, different behavior                       | method overriding, duck typing       |
| Abstraction   | Expose what, hide how; enforce contracts                 | `abc.ABC`, `@abstractmethod`         |

## 1. Class basics — `__init__`, `self`

In [ ]:
class Dog:
    # __init__ runs when you call Dog(...) — it builds the instance
    def __init__(self, name, age):
        self.name = name          # instance attribute — unique per object
        self.age = age

    def bark(self):               # instance method — first arg is always self
        return f'{self.name} says woof!'

d = Dog('Rex', 3)                 # __init__ called; d is the instance
print(d.name, d.age)              # Rex 3
print(d.bark())                   # Rex says woof!

# `self` is just a convention — it's the instance, passed implicitly
# d.bark()  is shorthand for  Dog.bark(d)

## 2. Instance vs class attributes

- **Instance attribute** → defined on `self`, unique per object.
- **Class attribute** → defined on the class, shared across all instances.

In [ ]:
class Dog:
    species = 'Canis familiaris'   # class attribute — shared

    def __init__(self, name):
        self.name = name           # instance attribute — per object

a, b = Dog('Rex'), Dog('Buddy')
print(a.species, b.species)        # both 'Canis familiaris'

Dog.species = 'Dog'                # change on the class → seen by all instances
print(a.species, b.species)        # both 'Dog'

a.species = 'Wolf'                 # assigning on instance creates a NEW instance attr
print(a.species, b.species)        # 'Wolf'  'Dog'  — shadows the class attr on `a`

# GOTCHA: mutable class attributes are shared!
class Bag:
    items = []                     # BAD — every Bag shares the same list
x, y = Bag(), Bag()
x.items.append('apple')
print(y.items)                     # ['apple']  ← surprise
# Fix: assign in __init__  →  self.items = []

## 3. Method types — instance / class / static

| Decorator       | First arg | Use when…                                       |
|-----------------|-----------|-------------------------------------------------|
| (none)          | `self`    | needs the instance                              |
| `@classmethod`  | `cls`     | needs the class (alt constructors, class state) |
| `@staticmethod` | —         | logically grouped helper; needs neither         |

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def to_fahrenheit(self):                     # instance method
        return self.celsius * 9/5 + 32

    @classmethod
    def from_fahrenheit(cls, f):                  # alternative constructor
        return cls((f - 32) * 5/9)                # cls = Temperature (or subclass)

    @staticmethod
    def is_freezing(celsius):                     # plain utility — no self / cls
        return celsius <= 0

t = Temperature.from_fahrenheit(100)
print(round(t.celsius, 2), t.to_fahrenheit())     # 37.78  100.0
print(Temperature.is_freezing(-5))                # True

### 3a. Mental model — `@classmethod`'s two faces

`@classmethod` has **two distinct uses**, decided by what it returns:

| If it returns…              | It's acting as…                       | Closest C++ analogue                        |
|-----------------------------|---------------------------------------|---------------------------------------------|
| `cls(...)` — a new instance | **Alternative / factory constructor** | Named constructor / static factory method  |
| something else (or nothing) | **Class-level state operation**       | Static method touching static members      |

> Returning `cls(...)` ⇒ constructor-ish; otherwise ⇒ acts on class-level (static) variables.

**Note vs. C++ copy ctor.** A copy ctor clones an *existing* instance of the same type. A `@classmethod` factory usually builds an instance from **different inputs** (dict, string, age, file). It *can* implement a copy ctor — but that's just one flavor of factory.

In [ ]:
# --- Face 1: factory (returns cls(...)) ---
from datetime import date

class Person:
    def __init__(self, name, birth_year):
        self.name, self.birth_year = name, birth_year

    @classmethod
    def from_birth_year(cls, name, age):           # factory — different inputs
        return cls(name, date.today().year - age)

    @classmethod
    def from_dict(cls, d):                          # another factory
        return cls(d['name'], d['birth_year'])

    @classmethod
    def copy_of(cls, other):                        # factory acting as copy ctor
        return cls(other.name, other.birth_year)

p1 = Person('Alice', 1990)
p2 = Person.from_birth_year('Bob', 30)
p3 = Person.from_dict({'name': 'Cara', 'birth_year': 2000})
p4 = Person.copy_of(p1)
print(p2.birth_year, p3.name, p4.name)

### 3b. Why `cls(...)` and not `Person(...)` — subclass-safety

Both faces of `@classmethod` need `cls` (not a hard-coded class name) for the same underlying reason: **so subclasses behave correctly.**

- **Factory case** → `cls(...)` builds an instance of the *actual* class the method was called on (subclass-aware).
- **Class-state case** → `cls.x` reads/writes the *actual* class's attribute, so subclasses can keep independent state.

In [ ]:
# --- Factory + subclass-safety ---
class Employee(Person):
    pass

e = Employee.copy_of(p1)
print(type(e).__name__)        # Employee — cls was Employee, not Person
# If copy_of had used `Person(...)` instead of `cls(...)`, type(e) would be Person.

# --- Face 2: class-state operation (does NOT return cls(...)) ---
class Counter:
    count = 0                  # class-level (static) variable

    @classmethod
    def increment(cls):
        cls.count += 1         # touches cls.count, not Counter.count directly

    @classmethod
    def reset(cls):
        cls.count = 0

class SubCounter(Counter):
    count = 0                  # own copy — independent from parent

Counter.increment(); Counter.increment()
SubCounter.increment()
print(Counter.count, SubCounter.count)   # 2 1  — independent because cls differs

## 4. Encapsulation — public / protected / private

Python has **no real private** — just naming conventions.

| Prefix | Meaning            | Enforced? |
|--------|--------------------|-----------|
| `name`   | public           | —         |
| `_name`  | protected (hint) | by convention only |
| `__name` | private (mangled to `_ClassName__name`) | partial — discourages access |

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner             # public
        self._note = 'internal'        # protected (hands off, by convention)
        self.__balance = balance       # name-mangled → _Account__balance

    def deposit(self, amount):
        self.__balance += amount

    def get_balance(self):
        return self.__balance

a = Account('Alice', 100)
print(a.owner, a._note, a.get_balance())
# print(a.__balance)            # AttributeError
print(a._Account__balance)      # works — mangling, not real hiding

## 5. `@property` — computed / validated attributes

Looks like an attribute on the outside, runs code on the inside. Use for **validation**, **derived values**, or **read-only** fields.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius           # triggers setter below

    @property
    def radius(self):                  # getter
        return self._radius

    @radius.setter
    def radius(self, value):           # setter — runs validation
        if value < 0:
            raise ValueError('radius must be >= 0')
        self._radius = value

    @property
    def area(self):                    # read-only derived attribute
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.radius, c.area)                # 5  78.53975
c.radius = 10                          # uses setter
# c.radius = -1                        # ValueError
# c.area = 100                         # AttributeError — no setter

## 6. Inheritance + `super()`

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f'{self.name} makes a sound'

class Dog(Animal):                    # Dog inherits from Animal
    def __init__(self, name, breed):
        super().__init__(name)        # call parent's __init__
        self.breed = breed

    def speak(self):                  # override
        return f'{self.name} barks'

d = Dog('Rex', 'Labrador')
print(d.name, d.breed, d.speak())
print(isinstance(d, Animal))          # True
print(issubclass(Dog, Animal))        # True

## 7. Multiple inheritance + MRO

Python resolves multiple parents using **C3 linearization** — inspect with `Class.__mro__`.

In [ ]:
class A:
    def greet(self): return 'A'
class B(A):
    def greet(self): return 'B'
class C(A):
    def greet(self): return 'C'
class D(B, C):                        # diamond
    pass

print(D().greet())                    # 'B' — leftmost parent wins
print([cls.__name__ for cls in D.__mro__])
# ['D', 'B', 'C', 'A', 'object']  — order Python searches

## 8. Polymorphism — same call, different behavior

**Method overriding** (inheritance-based) + **duck typing** ("if it quacks like a duck...").

In [ ]:
class Cat:
    def speak(self): return 'meow'
class Cow:
    def speak(self): return 'moo'
class Robot:
    def speak(self): return 'beep'

# Duck typing — no shared base class needed; just needs a .speak()
for animal in [Cat(), Cow(), Robot()]:
    print(animal.speak())

## 9. Abstraction — `abc.ABC` + `@abstractmethod`

Force subclasses to implement specific methods. Trying to instantiate an abstract class raises `TypeError`.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...               # subclass MUST implement

    def describe(self):               # concrete method — inherited as-is
        return f'A shape with area {self.area()}'

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

# Shape()        # TypeError — can't instantiate abstract class
print(Square(4).describe())           # A shape with area 16

## 10. Dunder (magic) methods

Define how your object behaves with built-ins (`print`, `==`, `len`, `+`, `for`, `in`, ...).

| Dunder     | Triggered by         |
|------------|----------------------|
| `__str__`  | `str(x)`, `print(x)` — user-friendly |
| `__repr__` | `repr(x)`, REPL display — unambiguous |
| `__eq__`   | `x == y`             |
| `__len__`  | `len(x)`             |
| `__add__`  | `x + y`              |
| `__iter__` | `for i in x`         |
| `__contains__` | `x in y`         |

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title, self.pages = title, pages

    def __repr__(self):  return f'Book({self.title!r}, {self.pages})'   # debug
    def __str__(self):   return f'"{self.title}" ({self.pages}p)'         # user
    def __len__(self):   return self.pages
    def __eq__(self, o): return isinstance(o, Book) and self.title == o.title
    def __add__(self, o): return Book(f'{self.title} + {o.title}', self.pages + o.pages)

b1 = Book('Dune', 412)
b2 = Book('Foundation', 255)
print(b1)                              # "Dune" (412p)        — __str__
print([b1])                            # [Book('Dune', 412)] — __repr__
print(len(b1), b1 == Book('Dune', 9))  # 412 True
print(b1 + b2)                         # "Dune + Foundation" (667p)

## 11. `@dataclass` — boilerplate killer

Auto-generates `__init__`, `__repr__`, `__eq__` from type-annotated fields.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float = 0.0                     # default
    tags: list = field(default_factory=list)   # mutable default → use factory

p = Point(1, 2)
print(p)                               # Point(x=1, y=2, tags=[])
print(p == Point(1, 2))                # True — __eq__ generated

# Frozen (immutable, hashable) version:
@dataclass(frozen=True)
class FrozenPoint:
    x: float
    y: float

fp = FrozenPoint(1, 2)
# fp.x = 9                             # FrozenInstanceError
print({fp})                            # works — hashable

## 12. `__slots__` — memory + attribute lockdown

Replaces the per-instance `__dict__` with a fixed slot table. **Less memory** and **prevents new attributes**.

In [ ]:
class Pixel:
    __slots__ = ('x', 'y')             # only x, y allowed
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Pixel(1, 2)
print(p.x, p.y)
# p.color = 'red'                      # AttributeError — no slot, no __dict__
# hasattr(p, '__dict__')               # False

## Cheat sheet

```python
class Foo(Base):                # inheritance
    class_attr = 0              # shared across instances
    __slots__ = ('a', 'b')      # optional — locks attrs, saves memory

    def __init__(self, a):      # constructor
        super().__init__()      # call parent
        self.a = a              # instance attr

    def method(self): ...       # instance method

    @classmethod
    def alt_ctor(cls, x): return cls(x)

    @staticmethod
    def helper(x): return x * 2

    @property
    def derived(self): return self.a * 10
```

**Rules of thumb**
- Use `@dataclass` for plain value holders.
- Use `ABC` + `@abstractmethod` to enforce a contract.
- Prefer composition over deep inheritance.
- Don't use `__private` unless you really need name mangling — `_protected` is usually enough.